# 🏟️ HaxBall A3 — 3v3 Multi-Agent Shared Policy Training
**Branch:** `MARL` | **Env:** `A3Env` | **Algorithm:** `MaskablePPO` (sb3_contrib)

Architecture: `SharedPolicyVecEnv` — 1 policy drives all 6 agents (3 red + 3 blue).

## 1. Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/NLightVN/haxball-agent-lite.git"
BRANCH   = "MARL"
REPO_DIR = "/content/haxball-agent-lite"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest...")
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Git identity (để push checkpoint commit nếu cần)

In [ ]:
GIT_NAME  = "NLightVN"
GIT_EMAIL = "hung24378un@gmail.com"

!git config user.name  "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
print("Git identity configured.")

## 3. Install requirements

In [ ]:
# sb3_contrib cần cho MaskablePPO (A3 dùng action masking)
!pip install -q stable-baselines3[extra] sb3-contrib gymnasium numpy torch

## 4. Mount Google Drive & tạo symlink checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Đường dẫn trên Drive ──────────────────────────────────────────────────────
DRIVE_BASE       = "/content/drive/MyDrive/haxball_A3_training"
DRIVE_A3_CKPT    = f"{DRIVE_BASE}/A3_checkpoints"   # checkpoint A3
DRIVE_A01_CKPT   = f"{DRIVE_BASE}/a0.1_checkpoints" # pretrained A0.1 (nếu có)

# Tạo thư mục trên Drive
!mkdir -p {DRIVE_A3_CKPT}
!mkdir -p {DRIVE_A01_CKPT}

# ── Symlink vào repo ──────────────────────────────────────────────────────────
LOCAL_MODELS = f"{REPO_DIR}/models"
!mkdir -p {LOCAL_MODELS}

# Xóa thư mục local cũ (nếu có) trước khi tạo symlink
!rm -rf {LOCAL_MODELS}/A3_checkpoints
!rm -rf {LOCAL_MODELS}/a0.1_checkpoints

# Symlink: models/A3_checkpoints  → Drive
!ln -s {DRIVE_A3_CKPT}  {LOCAL_MODELS}/A3_checkpoints
# Symlink: models/a0.1_checkpoints → Drive  (để load pretrained A0.1 weights)
!ln -s {DRIVE_A01_CKPT} {LOCAL_MODELS}/a0.1_checkpoints

print("Symlinks created:")
!ls -la {LOCAL_MODELS}/

## 5. Cấu hình training

In [ ]:
# ── Cấu hình ─────────────────────────────────────────────────────────────────
TOTAL_STEPS  = 80_000_000   # tổng số bước training

# Để resume từ checkpoint cụ thể, set số bước ở đây (ví dụ: 5_000_000)
# Set = 0 để train từ đầu (hoặc load từ A0.1 nếu có)
CONTINUE_STEP = 0

# Kiểm tra checkpoint có sẵn
import os, glob
ckpt_dir = f"{REPO_DIR}/models/A3_checkpoints"
ckpts = sorted(glob.glob(f"{ckpt_dir}/a3_*_steps.zip"))
print(f"Found {len(ckpts)} checkpoint(s) in Drive:")
for c in ckpts[-5:]:
    print(f"  {os.path.basename(c)}")

a01_best = f"{REPO_DIR}/models/a0.1_checkpoints/a0.1_best.zip"
print(f"\nA0.1 pretrained weights: {'✅ Found' if os.path.exists(a01_best) else '❌ Not found (will train from scratch)'}")

## 6. Training A3 (3v3 Shared Policy + Self-Play)

In [ ]:
import subprocess, sys

cmd = [sys.executable, f"{REPO_DIR}/training/train_a3.py"]
if CONTINUE_STEP > 0:
    cmd += ["--continue_step", str(CONTINUE_STEP)]

print("Running:", " ".join(cmd))
print("-" * 60)

# Chạy training (output stream trực tiếp)
!{sys.executable} {REPO_DIR}/training/train_a3.py {'--continue_step ' + str(CONTINUE_STEP) if CONTINUE_STEP > 0 else ''}

## 💡 Resume tips

Khi Colab bị disconnect, chỉ cần:
1. Chạy lại tất cả cells từ đầu
2. Trong cell **Cấu hình**, set `CONTINUE_STEP` = số step của checkpoint cuối cùng trên Drive
   - Ví dụ: nếu có `a3_5000000_steps.zip` → `CONTINUE_STEP = 5_000_000`
3. Cell **Training** sẽ tự load checkpoint đó và tiếp tục

Checkpoints được lưu tự động vào Google Drive theo lịch exponential:
`500K → 750K → 1.1M → 1.6M → 2.5M → ...`